In [1]:
import pandas as pd
#from pandas.io.parsers import ParserError
import numpy as np
from helper import get_mapper
import json
import os
import re

In [2]:
from os import listdir, stat
from os.path import isfile, join
BASE_DIR = "."
MIN_SIZE = 512

In [3]:
FOLDERS = [os.path.join(BASE_DIR, o) for o in os.listdir(BASE_DIR) if os.path.isdir(os.path.join(BASE_DIR,o))]
FOLDERS.sort()
FOLDERS = FOLDERS[1:-4]
YEARS = ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']

In [4]:
FOLDERS

['./2015',
 './2016',
 './2017',
 './2018',
 './2019',
 './2020',
 './2021',
 './2022',
 './2023',
 './2024',
 './2025']

In [5]:
#bpm = pd.read_csv("../basic/block_plant_mapper.csv")
#bpm['plantid'] = bpm['plantid'].apply(lambda x: str(x).replace('/', '_'))
#ssem = pd.read_csv("../basic/plant_sse_mapper.csv")
#ssem['plantid'] = ssem['plantid'].apply(lambda x: str(x).replace('/', '_'))

raw_sse = pd.read_csv("../basic/inspire_prtr_mapper.csv")
see = raw_sse.rename(columns={"InspireID_Betrieb": "plantid"})
seem = see[['plantid', 'sseid']]
#bpm['plantid'] = bpm['plantid'].apply(lambda x: str(x).replace('/', '_'))
seem['plantid'] = seem['plantid'].apply(lambda x: str(x).replace('/', '_'))

plantslist = list(set(seem['plantid'].to_list()))

/tmp/ipykernel_905000/4204571207.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  seem['plantid'] = seem['plantid'].apply(lambda x: str(x).replace('/', '_'))


In [6]:
seem

,plantid,sseid
0,06-02-B10117A007,SEE987197130805
1,06-02-B10117A007,SEE913896693631
2,06-05-100-0030723,BNA1084
3,06-05-100-0431554,BNA0992
4,06-05-100-0431554,BNA0991
...,...,...
1105,SD662-98,SEE927542617698
1106,SD662-98,SEE915847711449
1107,SD662-98,SEE912980761904
1108,SD662-98,SEE918332839635


In [7]:
#plantslist

In [8]:
def save2combined(df, plantname):
    df.to_csv("./combined/" + plantname + ".csv", index=False)

In [9]:
def squash_production(plantslist):
    emptyplants = []
    for plant in plantslist:
        #print(plant)
        dflist = []
        for year in YEARS:
            try:
                df = pd.read_csv("./by_plantid/" + year + "/" + plant + ".csv")
                #df.fillna(0, inplace=True)
                df[df.columns[1:]] = df[df.columns[1:]].astype(int)
            except FileNotFoundError:
                #print("./by_plantid/" + year + "/" + plant + ".csv")
                df = pd.DataFrame({'produced_at': []})
    
            testdf = df.dropna()
            if testdf.empty:
                emptyplants.append(plant)
            else:
                dflist.append(df)
                
        if len(dflist) > 0:
            newdf = pd.concat(dflist)
            save2combined(newdf, plant)

    return list(set(emptyplants))

In [10]:
a = squash_production(plantslist)

In [11]:
len(plantslist)

355

In [12]:
len(a)

332

In [13]:
errorset = []
for plant in plantslist:
    dflist = []
    for year in YEARS:
        df = pd.DataFrame()
        try:
            df = pd.read_csv("./by_plantid/" + year + "/" + plant + ".csv", na_values=['-'])
        except FileNotFoundError:
            if (plant not in errorset):
                errorset.append(plant)
            pass
        
        dflist.append(df)
    newdf = pd.concat(dflist)
    #print(newdf.shape)
print(errorset)

['NW300-0370387', 'SD661-06', 'BYS00045', 'BYS00471', 'SD661-97', 'SL0101003-G', 'RP5000656', 'NW300-0072412', 'HE20000115', 'SD661-44', 'BYS00114', 'NI06060116320', 'SN80011256', 'BYS00235', 'BYS00291', 'NW100-0042820', 'ST100032', 'SD661-85', 'BWpf-450-80920445-00000000', 'BYS00311', 'SD662-14', 'SD661-108', 'NI03267770260', 'NW900-0271161', 'NW700-0020290', 'BYS00727', 'BYS00233', 'SD661-92', 'BYS00334', 'SD661-112', 'SD661-81', 'NW300-0079450', 'BWpf-450-1383701-00000000', 'DE.EEA44484', 'DE.EEA43267', 'SD661-16', 'SN70015959', 'SD666-16', 'NI10285139970', 'BYS00925', 'BB16018798', 'SL0101000-G', 'SD661-105', 'NW500-9981505', 'BB45026224', 'SD661-84', 'SD661-111', 'NW900-9140178', 'NW500-0342658', 'BYS00282', 'BYS00869', 'SN80011250', 'SD661-103', 'SD662-01', 'SD661-39', 'NW900-0205103', 'SD661-107', 'HE50002167', 'TH30013152', 'SD661-80', 'DE.EEA44408', 'BYS00135', 'SN80011266', 'SD661-77', 'BYS00708', 'NI04247180070', 'HE50000118', 'ST100609', 'NI01010959490', 'NW300-0215443', '0

In [14]:
newdf

""


In [15]:
True if '06-05-500-0915123' in errorset else False

False